# Neural Technology API Walkthrough

Use this notebook to analyze a `torch.nn.Module` with the public MLIA Python API and the Neural Technology plugins.

It describes how MLIA can:

- pass a `torch.nn.Module` to `run_advisor(...)`;
- provide `example_inputs` for PT2 export;
- run Neural Technology compatibility analysis; and
- inspect the standardized output returned as a Python dictionary.

For path-based `.vgf`, `.tosa`, `.tflite`, `.pt2`, and `.pte` workflows, see the [CLI guide](cli.md).

## Run the notebook

Configure access to the package indexes required by `pyproject.toml`. Then, from the repository root, start Jupyter with the project and transient notebook dependencies:

```bash
uv run --with jupyterlab --with torch --with mlia-converters-pytorch jupyter lab docs/source/neural_technology_api_walkthrough.ipynb
```

The command installs this package in the environment, so MLIA discovers its target and backend entry points automatically.

## How module analysis works

Neural Technology participates in the core MLIA module-input workflow:

- The target advertises module-input support.
- Core MLIA exports the module to PT2 using `example_inputs`.
- The installed PyTorch converter prepares the model for Neural Technology analysis.
- `run_advisor(...)` returns standardized output in memory.


## Public API

The examples use public imports from `mlia`. Installed package entry points provide plugin discovery; no manual registry changes are required.

Use `list_targets()`, `list_backends()`, and `supported_backends()` to inspect the capabilities available in the notebook environment.


In [ ]:
from __future__ import annotations

import json
from pprint import pprint

from mlia import list_backends, list_targets, run_advisor, supported_backends

TARGET_PROFILE = "neural-technology"

In [ ]:
print("Neural Technology targets:")
pprint([item for item in list_targets() if item.get("target") == "neural-technology"])

print("\nSupported backends:")
pprint(supported_backends(TARGET_PROFILE))

## Inspect installed capabilities

Confirm that MLIA discovers this package before running analysis.


In [ ]:
print("Registered backends:")
pprint([backend["name"] for backend in list_backends()])

In [ ]:
target = next(
    item for item in list_targets() if item.get("target") == "neural-technology"
)
pprint(target)

## `torch.nn.Module` input with `run_advisor(...)`

This is the main flow the notebook is about. The module goes into the public core API directly, not into a target-specific helper.

MLIA will:

1. Export the module to PT2 using `example_inputs`.
2. Select the Neural Technology target and backend.
3. Return standardized output as a Python dictionary.


In [ ]:
def try_run_advisor(**kwargs):
    try:
        return run_advisor(**kwargs)
    except Exception as exc:
        print(f"{type(exc).__name__}: {exc}")
        return None


def summarize_result(result: dict | None) -> None:
    if result is None:
        print("No result returned.")
        return

    print(f"schema_version: {result.get('schema_version')}")
    print(f"model:          {result.get('model', {})}")
    print(f"context keys:   {sorted(result.get('context', {}).keys())}")
    print(
        f"backends:       {[item.get('name') for item in result.get('backends', [])]}"
    )
    print(f"result count:   {len(result.get('results', []))}")

In [ ]:
try:
    import torch
    import torch.nn as nn
except Exception as exc:
    print(f"Skipping module example because torch is unavailable: {exc}")
    module_result = None
else:

    class DemoModule(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv = nn.Conv2d(1, 4, kernel_size=3)
            self.relu = nn.ReLU()

        def forward(self, x):
            return self.relu(self.conv(x))

    module = DemoModule()
    example_inputs = (torch.randn(1, 1, 28, 28),)

    module_result = try_run_advisor(
        advice_category="compatibility",
        target_profile=TARGET_PROFILE,
        model=module,
        example_inputs=example_inputs,
        backends=["nx-performance-estimator"],
    )

    summarize_result(module_result)

In [ ]:
if module_result is not None:
    print(json.dumps(module_result, indent=2)[:2500])

## Inspect result categories

The public API returns each analysis result in the top-level `results` list.

Inspect the result kind, status, and producer before reading detailed metrics or checks.


In [ ]:
if module_result is not None:
    for result in module_result.get("results", []):
        pprint({key: result.get(key) for key in ("kind", "status", "producer")})

## Path-based model formats

The Neural Technology target also supports path-based model analysis.

TFLite and PT2 inputs rely on separate converter packages:

- `mlia-converters-litert` provides TFLite-to-TOSA conversion.
- `mlia-converters-pytorch` provides PT2-to-TOSA conversion.

With the required plugins installed:

- `.vgf` and `.tosa` use the package's direct analysis path.
- `.tflite` uses the LiteRT converter plugin.
- `.pt2` and `.pte` use the PyTorch converter plugin.
- `torch.nn.Module` inputs are exported to PT2 by the core API.


## Takeaways

- Use `run_advisor(...)` for `torch.nn.Module` analysis through core MLIA.
- Provide representative `example_inputs` for PT2 export.
- Use discovery helpers to confirm installed targets and backends.
- Install the matching converter plugin for `.tflite`, `.pt2`, and `.pte` inputs.
- Consume the standardized dictionary returned by the public API.
